## Task 1
Initialize the Feast Feature Repository

Set up a Feast feature repository within your existing IRIS pipeline project. Configure it to use a local backend (SQLite is sufficient). Ensure the repository structure follows Feast conventions and is committed to Git.

In [ ]:
feast init feature_repo

## Task 2
Define Entities, Data Sources & Feature Views

Within your feature repository, define:

An entity that uniquely identifies each iris sample. A data source pointing to your IRIS dataset. A feature view that maps the iris feature columns to the entity and data source.
Think carefully about what timestamp field Feast requires and how you will provide one for a static dataset like IRIS.

In [37]:
pwd

'/home/jupyter/IITMBS_22F1001871_MLOPS_WEEKLY_ASSIGNMENT'

In [40]:
#Read and print into to complete feature view and since feast can only handle parquet file, we need to update it. 
import pandas as pd

data=pd.read_csv("feature_repo/data/iris_data_adapted_for_feast.csv")

In [48]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   event_timestamp    45 non-null     datetime64[us, UTC]
 1   iris_id            45 non-null     int64              
 2   sepal_length       45 non-null     float64            
 3   sepal_width        45 non-null     float64            
 4   petal_length       45 non-null     float64            
 5   petal_width        45 non-null     float64            
 6   species            45 non-null     str                
 7   created_timestamp  45 non-null     datetime64[us, UTC]
dtypes: datetime64[us, UTC](2), float64(4), int64(1), str(1)
memory usage: 3.3 KB


In [47]:
#since event timestamp becomes an issue, we need to convert it to the datetime
data['event_timestamp'] = pd.to_datetime(data['event_timestamp'],utc=True)
data['created_timestamp'] = pd.to_datetime(data['created_timestamp'],utc=True)

In [49]:
data.to_parquet("feature_repo/data/iris.parquet", index=False)

## Task 3
Apply Definitions & Materialize Features

Register your feature definitions with the Feast registry, then materialize feature values into the online store. Verify that materialization completed successfully.

In [28]:
!feast apply

/bin/bash: line 1: feast: command not found


In [46]:
#to materalize we need to know the start and end timestamp of the dataset
print("Start date to materialize",min(data['event_timestamp']))
print("End date to materialize",max(data['event_timestamp']))

Start date to materialize 2025-09-17 10:40:17.102131+00:00
End date to materialize 2025-10-01 10:40:17.102131+00:00


In [ ]:
#to materalize i.e, to upload to the dataset to the online storage
!feast materialize "2025-09-17 10:40:17.102131+00:00" "2025-10-01 10:40:17.102131+00:00"

## Task 4
Fetch Features for Training (Offline Store)

Using Feast's offline retrieval, fetch historical features for your full training set. Train your IRIS classification model using the returned dataset. Your training code should pull features from Feast rather than reading the CSV directly — this is the key behavioral change.

In [9]:
import feast
import joblib 
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from feast import FeatureStore

store = FeatureStore(repo_path="feature_repo")

entity_df = data[["iris_id","event_timestamp"]]

print("Entity_Info")
print("------------------------\n\n")
entity_df.info()
print("------------------------\n\n")


training_df = store.get_historical_features(
    entity_df = entity_df,
    features = [
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
    ],
).to_df()

print("Retrived feature Info")
print("--------------------------\n\n")
training_df.info()
print("---------------------------\n\n")

X = training_df[
    [
        "sepal_length",
        "sepal_width",
        "petal_length",
        "petal_width",
    ]
]

y = data["species"]

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X,y)

joblib.dump(model,"artifacts/model.joblib")

print("Model Trained Successfully")

ModuleNotFoundError: No module named 'feast'